In [1]:
from moe_reft.olmoe.modeling_olmoe import OlmoeForCausalLM
from moe_reft.olmoe import configuration_olmoe
from moe_reft import interventions_config

model = OlmoeForCausalLM(configuration_olmoe.OlmoeInterventionsConfig(
    interventions_config=interventions_config.InterventionsConfig(
        intervention_places="after_moe",
        intervention_layers="even_only",
    ),
))

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
from __future__ import annotations

import torch
from loguru import logger
from torch import nn
from transformers import AutoModelForCausalLM, PreTrainedModel

map_dtype=torch.bfloat16  # optional casting
map_device=torch.device("cuda")  # optional device move
        
hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct"
# hf_model_name_or_path="allenai/OLMoE-1B-7B-0924-Instruct"

hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
        hf_model_name_or_path,
        dtype=map_dtype if map_dtype is not None else None,
        trust_remote_code=True,
    )

src_sd = hf_model.state_dict()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
from moe_reft.olmoe import load_weights

intervention_patterns = [
    "*.pre_moe_intervention.*",
    "*.after_moe_intervention.*",
    "*.pre_moe_intervenetion.*",  # typo fallback
]
    # 1) Load HF model & grab its state dict
hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    hf_model_name_or_path,
    dtype=map_dtype if map_dtype is not None else None,
    trust_remote_code="True",
)
src_sd = hf_model.state_dict()

# 2) Build filtered state dict compatible with your custom model
filtered_sd, report = load_weights.build_partial_state_dict(
    src_sd=src_sd,
    dst_module=model,
    intervention_patterns=intervention_patterns,
    device=map_device,
    dtype=map_dtype,
)

# 3) Load with strict=False (so missing keys — e.g., interventions — are fine)
missing, unexpected = model.load_state_dict(filtered_sd, strict=False)

# Merge loader feedback into the report
report.skipped_missing.extend(missing)
if unexpected:
    report.skipped_missing.extend(unexpected)

# 4) Freeze everything, then unfreeze only intervention layers
for param in model.parameters():
    param.requires_grad = False


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Building partial state dict: 100%|██████████| 3219/3219 [00:03<00:00, 1049.63it/s]


In [6]:
for name, param in model.named_parameters():
    if load_weights._matches_any(name, intervention_patterns):
        param.requires_grad = True

# 5) Print parameter stats
total_params, trainable_params = load_weights._count_parameters(model)
print(f"Total parameters:     {total_params}")
print(f"Trainable parameters: {trainable_params}")

logger.info(f"Parameter stats — total: {total_params}, trainable: {trainable_params}")

2025-11-08 18:03:22.956 | INFO     | __main__:<module>:10 - Parameter stats — total: 6919424064, trainable: 262208


Total parameters:     6919424064
Trainable parameters: 262208


In [10]:
for name,param in model.named_parameters():
    if load_weights._matches_any(name, intervention_patterns):
        print(name, param.shape)

model.layers.0.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.0.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.0.after_moe_intervention.learned_source.bias torch.Size([8])
model.layers.2.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.2.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.2.after_moe_intervention.learned_source.bias torch.Size([8])
model.layers.4.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.4.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.4.after_moe_intervention.learned_source.bias torch.Size([8])
model.layers.6.after_moe_intervention.rotate_layer.parametrizations.weight.original torch.Size([2048, 8])
model.layers.6.after_moe_intervention.learned_source.weight torch.Size([8, 2048])
model.layers.6.after_moe_i

In [ ]:
from moe_reft import read_config
from moe_reft import interventions_config, tiny_sft
from moe_reft.olmoe import modeling_olmoe, configuration_olmoe, load_weights

config_path = "moe_reft/configs/olmoe.yaml"

train_config, interventions_config_, olmoe_config = read_config.load_all_configs(config_path)

custom_model = modeling_olmoe.OlmoeForCausalLM(
    configuration_olmoe.OlmoeInterventionsConfig(interventios_config=interventions_config_)
)



/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


NameError: name 'torch' is not defined

In [ ]:
import torch
from loguru import logger
# 2) Load HF weights into the overlapping parts, skipping interventions
report = load_weights.load_hf_into_custom_model(
    hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct",
    custom_model=custom_model,
    intervention_patterns=["*.pre_moe_intervention.*", "*.after_moe_intervention.*"],
    map_dtype=torch.bfloat16,  # optional casting
    map_device=torch.device("cuda"),  # optional device move
    trust_remote_code=False,
)
logger.info(f"{report.summary()}")

for name, param in custom_model.named_parameters():
    if load_weights._matches_any(name, interventions_config.INTERVENTION_PATTERNS):
        param.requires_grad = True

# 5) Print parameter stats
total_params, trainable_params = load_weights._count_parameters(custom_model)
print(f"Total parameters:     {total_params}")
print(f"Trainable parameters: {trainable_params}")

logger.info(f"Parameter stats — total: {total_params}, trainable: {trainable_params}")
dataloader, _ = tiny_sft.build_tiny_sft_dataloader(model_name="allenai/OLMoE-1B-7B-0125-Instruct")

2025-11-08 20:17:23.894 | INFO     | __main__:<module>:12 - Copied: 3219 | Skipped (shape): 0 | Skipped (missing): 96 | Skipped (intervention): 0
2025-11-08 20:17:23.960 | INFO     | __main__:<module>:23 - Parameter stats — total: 6919686272, trainable: 524416


Total parameters:     6919686272
Trainable parameters: 524416


In [7]:
from moe_reft import train

for step, batch in enumerate(dataloader):
    model_inputs, labels = train._unpack_batch(batch, torch.device("cuda" ))
    break

In [1]:
from transformers import AutoTokenizer

prompt_template = [{"role":"system","content":"<SYSTEMT>"},{"role":"user","content":"<USER>"},{"role":"assistant","content":"<ASSISTANT>"}]

tokenizer = AutoTokenizer.from_pretrained("allenai/OLMoE-1B-7B-0125-Instruct")

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
chat_message = tokenizer.apply_chat_template(prompt_template, tokenize=False)

In [4]:
print(tokenizer.chat_template)

{{ bos_token }}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>
' + message['content'] + '
' }}{% elif message['role'] == 'user' %}{{ '<|user|>
' + message['content'] + '
' }}{% elif message['role'] == 'assistant' %}{% if not loop.last %}{{ '<|assistant|>
'  + message['content'] + eos_token + '
' }}{% else %}{{ '<|assistant|>
'  + message['content'] + eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|assistant|>
' }}{% endif %}{% endfor %}


In [5]:
tokenizer.bos_token

'|||IP_ADDRESS|||'

In [6]:
chat_message

'|||IP_ADDRESS|||<|system|>\n<SYSTEMT>\n<|user|>\n<USER>\n<|assistant|>\n<ASSISTANT>|||IP_ADDRESS|||'

In [8]:
tokenizer.apply_chat_template(prompt_template[:-1], tokenizer=False, add_generation_prompt=True,tokenize=False)

'|||IP_ADDRESS|||<|system|>\n<SYSTEMT>\n<|user|>\n<USER>\n<|assistant|>\n'

In [11]:
import re

def extract_user_segment(text: str) -> str | None:
    match = re.search(r"<USER>(.*?)<ASSISTANT>", text, flags=re.DOTALL)
    if not match:
        return None
    # Extract raw segment
    segment = match.group(1)
    # Remove any leading non-alphanumeric characters
    # Trim trailing spaces/newlines
    return segment


extract_user_segment(chat_message)

'\n<|assistant|>\n'

In [13]:
tokenizer.apply_chat_template([{"role":"assistant","content":"<ASSISTANT>"}], tokenize=False)

'|||IP_ADDRESS|||<|assistant|>\n<ASSISTANT>|||IP_ADDRESS|||'

In [17]:
from typing import List

def find_subsequence(haystack: List[int], needle: List[int]) -> int:
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i : i + m] == needle:
            return i + m  # end index (1-based like your example)
    return -1

# Example
input_ids = [1, 2, 3,8,3,4, 5, 6]
response = [3, 4]

idx = find_subsequence(input_ids, response)
print(idx)  # 4


6


In [18]:
input_ids[6:]

[5, 6]

In [21]:
tokenizer.encode("<ASSISTANT>") + [tokenizer.eos_token_id]

[29, 1719, 5824, 1267, 5656, 31, 50279]

In [22]:
import re
from typing import Any, Callable, Mapping, Optional, Protocol, Mapping, cast
from loguru import logger
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    PreTrainedTokenizerBase,
)

CROSS_ENTROPY_IGNORE_INDEX = -100
PROMPT_TEMPLATE = [
    {"role": "system", "content": "<SYSTEMT>"},
    {"role": "user", "content": "<USER>"},
    {"role": "assistant", "content": "<ASSISTANT>"},
]


def extract_response_template(tokenizer: PreTrainedTokenizerBase) -> str | None:
    text = cast(str, tokenizer.apply_chat_template(PROMPT_TEMPLATE, tokenize=False))
    match = re.search(r"<USER>(.*?)<ASSISTANT>", text, flags=re.DOTALL)
    if not match:
        return None
    segment = match.group(1)
    return segment


class Transform(Protocol):
    """
    Loose interface for all data and model transforms. Transforms operate at the
    sample level and perform operations on a sample dict, returning the updated dict.
    For an example implementation of this protocol, see
    :class:`~torchtune.modules.transforms.VisionCrossAttentionMask`.
    """

    def __call__(self, sample: Mapping[str, Any]) -> Mapping[str, Any]:
        pass


class SFTDataset(Dataset):
    def __init__(
        self,
        *,
        source: str,
        tokenizer_model_name: str,
        train_on_input: bool = False,
        system_key: str | None,
        user_key: str,
        assistant_key: str,
        system_message: str | None,
        filter_fn: Optional[Callable] = None,
        filter_kwargs: Optional[dict[str, Any]] = None,
        **load_dataset_kwargs: dict[str, Any],
    ) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
        self.response_template = extract_response_template(self.tokenizer)
        logger.info(
            f"For the {tokenizer_model_name=} automatically assigned the response template to {self.response_template}"
        )
        self._data = load_dataset(source, **load_dataset_kwargs)
        if filter_fn is not None:
            if filter_kwargs is None:
                filter_kwargs = {}
            self._data = self._data.filter(filter_fn, **filter_kwargs)
        self._prepare_sample = SFTTransform(
            tokenizer=self.tokenizer,
            response_template=self.response_template,
            system_message=system_message,
            system_key=system_key,
            user_key=user_key,
            assistant_key=assistant_key,
            train_on_input=train_on_input,
        )

    def __len__(self):
        return len(self._data)

    def __getitem__(self, index: int) -> dict[str, Any]:
        sample = self._data[index]
        return self._prepare_sample(sample)


def _find_subseq(h: list[int], n: list[int]) -> int:
    for i in range(len(h) - len(n) + 1):
        if h[i : i + len(n)] == n:
            return i
    return -1


class SFTTransform(Transform):
    def __init__(
        self,
        tokenizer: PreTrainedTokenizerBase,
        response_template: str,
        system_key: str | None,
        user_key: str,
        assistant_key: str,
        train_on_input: bool,
        system_message: str | None,
    ):
        self.tokenizer = tokenizer
        if system_key and system_message:
            raise ValueError(
                f"Can't set both `system_message` and `system_key`, but they are set {system_key=} and {system_message=}"
            )
        self.system_key = system_key
        self.system_message = system_message
        self.user_key = user_key
        self.assistant_key = assistant_key
        self.response_template = response_template
        self.response_template_ids = self.tokenizer.encode(self.response_template)
        self.train_on_input = train_on_input

    def __call__(self, sample: Mapping[str, Any]) -> dict[str, Any]:
        system_message = self.system_message if self.system_message else sample[self.system_key]
        user_message = sample[self.user_key]
        assistant_message = sample[self.assistant_key]
        assert system_message
        assert user_message
        assert assistant_message

        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        enc = self.tokenizer(
            text,
            add_special_tokens=False,
        )
        input_ids: list[int] = enc["input_ids"]
        attention_mask: list[int] = enc["attention_mask"]

        labels = [-100] * len(input_ids)

        start = _find_subseq(input_ids, self.response_template_ids)
        if start != -1:
            start += len(self.response_template_ids)
            labels[start:] = input_ids[start:]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

